In [1]:
import pandas as pd

In [2]:
augmented_csv = "confidence_rank_augmented_temp.csv"
unique_edges_csv = "confidence_rank_unique_temp.csv"

In [3]:
df = pd.read_csv(augmented_csv, dtype=str)

In [4]:
# ------------------------------------------------------------------
# Remove rows marked as wrong
# ------------------------------------------------------------------

df = df[df["Direct"] != "wrong"].copy()

# ------------------------------------------------------------------
# Replace ambiguous signs with positive
# ------------------------------------------------------------------

ambiguous = df["Sign"] == "ambiguous"

if ambiguous.any():
    print("Rows with ambiguous Sign:\n")
    print(df.loc[ambiguous])
    print()

    df.loc[ambiguous, "Sign"] = "positive"

# ------------------------------------------------------------------
# Group by Regulator and Target
# ------------------------------------------------------------------

cols_that_should_match = [
    "Sign",
    "Confidence Rank",
    "Constraint",
    "Direct",
]

rows = []

for (reg, tar), group in df.groupby(["Regulator", "Target"], sort=True):

    # Check consistency
    inconsistent = False
    for col in cols_that_should_match:
        values = group[col].fillna("").unique()
        if len(values) > 1:
            print(f"Inconsistency for edge {reg} -> {tar}")
            print(f"  {col}: {list(values)}")
            inconsistent = True

    # Collect models
    models = sorted(group["Model"].dropna().unique())

    # Build merged row
    row = {
        "Regulator": reg,
        "Target": tar,
        "Sign": group.iloc[0]["Sign"],
        "Model": ", ".join(models),
        "Confidence Rank": group.iloc[0]["Confidence Rank"],
        "Constraint": group.iloc[0]["Constraint"],
        "Direct": group.iloc[0]["Direct"],
    }

    rows.append(row)

merged_df = pd.DataFrame(rows)

# ------------------------------------------------------------------
# Sort
# ------------------------------------------------------------------

merged_df = (
    merged_df
    .sort_values(by=list(merged_df.columns))
    .reset_index(drop=True)
)


Rows with ambiguous Sign:

    Regulator Target       Sign Model Confidence Rank Constraint  Direct  \
510      NFKB    IL2  ambiguous   N10               1  regulates  direct   

                                                 Notes  
510  treat as positive. NF-κB family members—especi...  



In [5]:
print(f"Original rows : {len(df)}")
print(f"Merged rows   : {len(merged_df)}")

merged_df.head()

Original rows : 846
Merged rows   : 430


,Regulator,Target,Sign,Model,Confidence Rank,Constraint,Direct
0,APC,CD28,positive,"AJ, N10",1,regulates,direct
1,APC,TCR,positive,"AJ, N10",1,necessary,direct
2,BCL6,GATA3,negative,"AJ, MS15",1,regulates,direct
3,BCL6,IFNG,negative,MS15,1,regulates,direct
4,BCL6,IFNG_2,negative,MS15,1,regulates,direct


In [6]:
merged_df.to_csv(unique_edges_csv, index=False)